In [ ]:
%pip install numpy pandas matplotlib seaborn scikit-learn joblib

# 🤖 Lucy College LMS — ML Model Status & Metrics

This notebook provides a complete overview of the student performance prediction model.

## 📋 Contents
1. Load Model & Data
2. Dataset Overview
3. Model Info & Hyperparameters
4. Confusion Matrix (TP, FP, TN, FN)
5. Core Metrics (Accuracy, Precision, Recall, F1, AUC)
6. Error Rates (FPR, FNR, Type I, Type II)
7. ROC Curve
8. Cross-Validation Scores
9. Feature Importance
10. Per-Class Metrics
11. Feature Statistics

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    roc_curve, roc_auc_score, precision_recall_curve
)
import joblib

# Config
MODEL_DIR = Path('models')
MODEL_PATH = MODEL_DIR / 'performance_model.joblib'
FEATURES_PATH = MODEL_DIR / 'feature_stats.json'
CSV_PATH = Path('lms_advanced_dataset.csv')
FEATURE_COLUMNS = [
    'attendance', 'quiz_score', 'participation', 'video_watch',
    'ppt_progress', 'has_video', 'has_ppt', 'assignment_score', 'course_type_encoded'
]
COURSE_TYPE_MAP = {'f2f': 0, 'online': 1, 'blended': 2}

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
print('✅ Imports loaded')import sys
print(sys.executable)

ModuleNotFoundError: No module named 'numpy'

## 1️⃣ Load Model & Data

In [ ]:
# Load model
assert MODEL_PATH.exists(), '❌ Model not trained yet. Run POST /ml/train first'
model = joblib.load(MODEL_PATH)
mtime = datetime.fromtimestamp(MODEL_PATH.stat().st_mtime).strftime('%Y-%m-%d %H:%M:%S')
size_kb = MODEL_PATH.stat().st_size / 1024
print(f'✅ Model loaded: {MODEL_PATH} ({size_kb:.1f} KB, trained {mtime})')

# Load CSV
assert CSV_PATH.exists(), '❌ CSV dataset not found'
df = pd.read_csv(CSV_PATH)
df['course_type_encoded'] = df['course_type'].map(COURSE_TYPE_MAP).fillna(1).astype(int)
df['has_video'] = df['has_video'].astype(int)
df['has_ppt'] = df['has_ppt'].astype(int)
df['pass'] = df['pass'].astype(int)
for col in FEATURE_COLUMNS:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# Same split as training
X = df[FEATURE_COLUMNS].values
y = df['pass'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(f'✅ Data loaded: {len(df)} records, {len(X_train)} train, {len(X_test)} test')

## 2️⃣ Dataset Overview

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Pass/Fail pie
pass_counts = df['pass'].value_counts()
labels = ['FAIL', 'PASS']
colors = ['#e74c3c', '#2ecc71']
ax1.pie(pass_counts.sort_index(), labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
ax1.set_title('Pass/Fail Distribution', fontsize=14, fontweight='bold')

# Course type pie
ct_counts = df['course_type'].value_counts()
ax2.pie(ct_counts, labels=ct_counts.index, autopct='%1.1f%%', startangle=90)
ax2.set_title('Course Type Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Total records: {len(df)}')
print(f'Pass: {int(df["pass"].sum())} ({df["pass"].mean()*100:.1f}%)')
print(f'Fail: {int(len(df) - df["pass"].sum())} ({(1-df["pass"].mean())*100:.1f}%)')

## 3️⃣ Model Info & Hyperparameters

In [ ]:
info = {
    'Algorithm': type(model).__name__,
    'N Estimators': model.n_estimators,
    'Max Depth': model.max_depth,
    'Random State': model.random_state,
    'Class Weight': model.class_weight,
    'N Features': model.n_features_in_,
    'Classes': model.classes_.tolist(),
    'Trained At': mtime,
    'File Size (KB)': f'{size_kb:.1f}',
    'Training Samples': len(X_train),
    'Test Samples': len(X_test),
}
for k, v in info.items():
    print(f'  {k:20s}: {v}')

## 4️⃣ Confusion Matrix (TP, FP, TN, FN)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
total = tp + fp + fn + tn

# Visual confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
labels_x = ['Predicted FAIL', 'Predicted PASS']
labels_y = ['Actual FAIL', 'Actual PASS']
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels_x, yticklabels=labels_y, ax=ax)
ax.set_title('Confusion Matrix', fontsize=16, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.show()

# Breakdown
print(f'\n🔢 Detailed Breakdown:')
print(f'  True Positive  (TP) : {tp:5d}  ({tp/total*100:5.2f}%)  — correctly predicted PASS')
print(f'  True Negative  (TN) : {tn:5d}  ({tn/total*100:5.2f}%)  — correctly predicted FAIL')
print(f'  False Positive (FP) : {fp:5d}  ({fp/total*100:5.2f}%)  — predicted PASS, actually FAIL (Type I)')
print(f'  False Negative (FN) : {fn:5d}  ({fn/total*100:5.2f}%)  — predicted FAIL, actually PASS (Type II)')

## 5️⃣ Core Metrics

In [ ]:
accuracy = (tp + tn) / total
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
auc = roc_auc_score(y_test, y_prob)

metrics = {
    'Accuracy': accuracy * 100,
    'Precision': precision * 100,
    'Recall (TPR)': recall * 100,
    'Specificity (TNR)': specificity * 100,
    'F1 Score': f1 * 100,
    'ROC AUC': auc * 100,
}

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(list(metrics.keys()), list(metrics.values()), color=['#3498db', '#2ecc71', '#e67e22', '#9b59b6', '#1abc9c', '#e74c3c'])
ax.set_xlim(0, 100)
ax.set_title('Core Model Metrics (%)', fontsize=16, fontweight='bold')
for bar, val in zip(bars, metrics.values()):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2, f'{val:.2f}%', va='center', fontsize=11)
plt.tight_layout()
plt.show()

for k, v in metrics.items():
    print(f'  {k:20s}: {v:.2f}%')

## 6️⃣ Error Rates

In [ ]:
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Error vs Correct
categories = ['True Positive', 'True Negative', 'False Positive\n(Type I)', 'False Negative\n(Type II)']
values = [tp, tn, fp, fn]
colors = ['#2ecc71', '#27ae60', '#e74c3c', '#c0392b']
ax1.bar(categories, values, color=colors)
ax1.set_title('Prediction Breakdown', fontsize=14, fontweight='bold')
for i, v in enumerate(values):
    ax1.text(i, v + 3, f'{v}\n({v/total*100:.1f}%)', ha='center', fontsize=10)
ax1.set_ylabel('Count')

# Error rates
error_labels = ['False Positive Rate\n(FPR)', 'False Negative Rate\n(FNR)']
error_vals = [fpr * 100, fnr * 100]
ax2.bar(error_labels, error_vals, color=['#e74c3c', '#c0392b'])
ax2.set_title('Error Rates (%)', fontsize=14, fontweight='bold')
ax2.set_ylim(0, max(error_vals) * 1.5)
for i, v in enumerate(error_vals):
    ax2.text(i, v + 1, f'{v:.2f}%', ha='center', fontsize=12)

plt.tight_layout()
plt.show()

print(f'\n📈 Error Rates:')
print(f'  False Positive Rate (FPR) : {fpr*100:.2f}%  — % of FAIL students wrongly flagged as PASS')
print(f'  False Negative Rate (FNR) : {fnr*100:.2f}%  — % of PASS students wrongly flagged as FAIL')
print(f'  Type I Error  : {fp} students told they\'ll pass but won\'t')
print(f'  Type II Error : {fn} students told they\'ll fail but would pass')

## 7️⃣ ROC Curve

In [ ]:
fpr_curve, tpr_curve, thresholds = roc_curve(y_test, y_prob)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr_curve, tpr_curve, color='#3498db', lw=2, label=f'ROC Curve (AUC = {auc:.4f})')
ax.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', label='Random Classifier')
ax.fill_between(fpr_curve, tpr_curve, alpha=0.15, color='#3498db')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve — Student Performance Prediction', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8️⃣ Cross-Validation Scores

In [ ]:
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')

fig, ax = plt.subplots(figsize=(8, 5))
folds = [f'Fold {i+1}' for i in range(5)]
bars = ax.bar(folds, cv_scores * 100, color=['#3498db', '#2ecc71', '#e67e22', '#9b59b6', '#1abc9c'])
ax.axhline(y=cv_scores.mean() * 100, color='red', linestyle='--', lw=2, label=f'Mean: {cv_scores.mean()*100:.2f}%')
ax.set_ylim(min(cv_scores * 100) - 5, 100)
ax.set_title('5-Fold Cross Validation Accuracy', fontsize=14, fontweight='bold')
ax.set_ylabel('Accuracy (%)')
for bar, val in zip(bars, cv_scores * 100):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.5, f'{val:.2f}%', ha='center', fontsize=10)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f'Fold scores: {[round(s, 4) for s in cv_scores]}')
print(f'Mean: {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%')

## 9️⃣ Feature Importance

In [ ]:
importance = dict(zip(FEATURE_COLUMNS, model.feature_importances_.tolist()))
sorted_imp = sorted(importance.items(), key=lambda x: x[1])
feat_names = [x[0] for x in sorted_imp]
feat_vals = [x[1] for x in sorted_imp]

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(feat_names)))
bars = ax.barh(feat_names, feat_vals, color=colors)
ax.set_xlabel('Importance', fontsize=12)
ax.set_title('Feature Importance (Random Forest)', fontsize=14, fontweight='bold')
for bar, val in zip(bars, feat_vals):
    ax.text(val + 0.003, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

print('Ranked Feature Importance:')
for i, (feat, val) in enumerate(sorted(importance.items(), key=lambda x: -x[1]), 1):
    print(f'  {i}. {feat:22s} {val:.4f}')

## 🔟 Per-Class Metrics

In [ ]:
report = classification_report(y_test, y_pred, target_names=['FAIL', 'PASS'], output_dict=True, zero_division=0)

classes = ['FAIL', 'PASS']
metric_names = ['precision', 'recall', 'f1-score']

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(metric_names))
width = 0.35

fail_vals = [report['FAIL'][m] * 100 for m in metric_names]
pass_vals = [report['PASS'][m] * 100 for m in metric_names]

bars1 = ax.bar(x - width/2, fail_vals, width, label='FAIL', color='#e74c3c', alpha=0.8)
bars2 = ax.bar(x + width/2, pass_vals, width, label='PASS', color='#2ecc71', alpha=0.8)

ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('Per-Class Metrics', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([m.title() for m in metric_names])
ax.set_ylim(0, 100)
ax.legend(fontsize=11)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{bar.get_height():.1f}%', ha='center', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{bar.get_height():.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

for cls in classes:
    m = report[cls]
    print(f'  {cls:5s}: precision={m["precision"]*100:.1f}%  recall={m["recall"]*100:.1f}%  f1={m["f1-score"]*100:.1f}%  support={m["support"]}')

## 1️⃣1️⃣ Feature Statistics

In [ ]:
if FEATURES_PATH.exists():
    with open(FEATURES_PATH) as f:
        stats = json.load(f)
    
    stat_df = pd.DataFrame(stats).T
    stat_df = stat_df[['mean', 'std', 'min', 'max']]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    stat_df['mean'].plot(kind='barh', xerr=stat_df['std'], ax=ax, color='#3498db', alpha=0.8, capsize=3)
    ax.set_xlabel('Value', fontsize=12)
    ax.set_title('Feature Mean ± Std Dev', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print('\nFeature Statistics Table:')
    display(stat_df.round(2))
else:
    print('Feature stats file not found')

## ✅ Summary

In [ ]:
print('='*60)
print('  LUCY ML MODEL SUMMARY')
print('='*60)
print(f'  Algorithm     : {type(model).__name__}')
print(f'  Accuracy      : {accuracy*100:.2f}%')
print(f'  ROC AUC       : {auc:.4f}')
print(f'  TP / FP / TN / FN : {tp} / {fp} / {tn} / {fn}')
print(f'  FPR (Type I)  : {fpr*100:.2f}%')
print(f'  FNR (Type II) : {fnr*100:.2f}%')
print(f'  5-Fold CV     : {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%')
print(f'  Top Feature   : {sorted(importance.items(), key=lambda x: -x[1])[0][0]}')
print(f'  Status        : ✅ Model trained and ready')
print('='*60)